# Feature Engineering Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys

# Allow imports from the project root
sys.path.insert(0, '..')

%matplotlib inline

In [ ]:
from src.features.temporal import add_temporal_features

# Create a sample DataFrame with dates
sample_dates = pd.DataFrame({
    'date': pd.date_range('2017-01-01', periods=30, freq='D'),
    'sales': np.random.rand(30) * 100,
    'store_nbr': 1,
    'family': 'GROCERY I'
})

sample_with_temporal = add_temporal_features(sample_dates, date_col='date')
temporal_cols = [c for c in sample_with_temporal.columns if c not in sample_dates.columns]
print(f'Temporal features added ({len(temporal_cols)}): {temporal_cols}')
display(sample_with_temporal[temporal_cols].head())

In [ ]:
# Visualize cyclical features: dow_sin and dow_cos
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(sample_with_temporal['dow_sin'], sample_with_temporal['dow_cos'],
                c=sample_with_temporal['day_of_week'], cmap='tab10', s=80)
axes[0].set_title('Cyclical Day-of-Week Encoding')
axes[0].set_xlabel('dow_sin')
axes[0].set_ylabel('dow_cos')
axes[0].set_aspect('equal')

axes[1].scatter(sample_with_temporal['month_sin'], sample_with_temporal['month_cos'],
                c=sample_with_temporal['month'], cmap='tab10', s=80)
axes[1].set_title('Cyclical Month Encoding')
axes[1].set_xlabel('month_sin')
axes[1].set_ylabel('month_cos')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
from src.features.lag_features import add_lag_features, add_rolling_features

# Build a longer sample for lag features
long_sample = pd.DataFrame({
    'date': pd.date_range('2016-01-01', periods=90, freq='D'),
    'sales': np.random.rand(90) * 100,
    'store_nbr': 1,
    'family': 'GROCERY I'
})

long_with_lags = add_lag_features(long_sample, target_col='sales')
lag_cols = [c for c in long_with_lags.columns if 'lag' in c]
print(f'Lag features added ({len(lag_cols)}): {lag_cols}')

long_with_rolling = add_rolling_features(long_with_lags, target_col='sales')
rolling_cols = [c for c in long_with_rolling.columns if 'rolling' in c or 'roll' in c]
print(f'Rolling features added ({len(rolling_cols)}): {rolling_cols}')

In [ ]:
from src.features.promotion import add_promotion_features

promo_sample = pd.DataFrame({
    'date': pd.date_range('2016-01-01', periods=60, freq='D'),
    'sales': np.random.rand(60) * 100,
    'store_nbr': 1,
    'family': 'GROCERY I',
    'onpromotion': np.random.choice([0, 1], 60, p=[0.7, 0.3])
})

promo_result = add_promotion_features(promo_sample)
promo_cols = [c for c in promo_result.columns if c not in promo_sample.columns]
print(f'Promotion features added ({len(promo_cols)}): {promo_cols}')
display(promo_result[['onpromotion'] + promo_cols].head(10))

In [ ]:
from src.features.cross_features import add_cross_features

cross_sample = pd.DataFrame({
    'date': pd.date_range('2016-01-01', periods=30, freq='D'),
    'sales': np.random.rand(30) * 100,
    'store_nbr': 1,
    'family': 'GROCERY I',
    'store_type': 'A',
    'cluster': 1,
    'day_of_week': pd.date_range('2016-01-01', periods=30, freq='D').dayofweek
})

cross_result = add_cross_features(cross_sample)
cross_cols = [c for c in cross_result.columns if c not in cross_sample.columns]
print(f'Cross features added ({len(cross_cols)}): {cross_cols}')
display(cross_result[cross_cols].head())

In [ ]:
feature_categories = {
    'Temporal': ['day', 'day_of_week', 'month', 'year', 'week_of_year', 'quarter',
                  'is_weekend', 'is_month_start', 'is_month_end',
                  'dow_sin', 'dow_cos', 'month_sin', 'month_cos'],
    'Lag': [f'sales_lag_{d}' for d in [1, 7, 14, 28]],
    'Rolling': [f'sales_rolling_mean_{w}' for w in [7, 14, 28, 90]] +
               [f'sales_rolling_std_{w}' for w in [7, 14, 28, 90]],
    'External': ['oil_price', 'oil_price_ma7', 'oil_price_lag1'],
    'Promotion': ['promo_lag_1', 'promo_lag_7', 'promo_rolling_7', 'promo_duration'],
    'Cross': ['family_x_store_type', 'family_x_cluster', 'dow_x_family'],
    'Aggregation': ['store_mean_sales', 'family_mean_sales', 'cluster_mean_sales']
}

total = 0
for category, features in feature_categories.items():
    print(f'{category}: {len(features)} features')
    total += len(features)
print(f'\nTotal features: {total}')

## Summary
Total feature categories: temporal, lag/rolling, external, promotion, cross, aggregation